In [4]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [5]:
train_data = pd.read_csv("./data/samsum-train.csv")
validation_data = pd.read_csv("./data/samsum-validation.csv")

In [6]:
train_data.head

<bound method NDFrame.head of              id                                           dialogue  \
0      13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1      13728867  Olivia: Who are you voting for in this electio...   
2      13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3      13730747  Edward: Rachel, I think I'm in ove with Bella....   
4      13728094  Sam: hey  overheard rick say something\r\nSam:...   
...         ...                                                ...   
14727  13863028  Romeo: You are on my ‘People you may know’ lis...   
14728  13828570  Theresa: <file_photo>\r\nTheresa: <file_photo>...   
14729  13819050  John: Every day some bad news. Japan will hunt...   
14730  13828395  Jennifer: Dear Celia! How are you doing?\r\nJe...   
14731  13729017  Georgia: are you ready for hotel hunting? We n...   

                                                 summary  
0      Amanda baked cookies and will bring Jerry some...  
1      Oliv

In [7]:
train_data.shape

(14732, 3)

In [9]:
validation_data.shape

(818, 3)

In [10]:
# random sampling and reduce data for now(not compulsary)
train_data = train_data.sample(n=5000, random_state = 42).reset_index(drop=True)
compulsoryvalidation_data = validation_data.sample(n=500, random_state = 42).reset_index(drop=True)

In [11]:
# Pre-processing Data 
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ",  text)  # remove next line etc.
    text = re.sub(r"s+", " ", text)  # extra space
    text = re.sub(r"<.*?>", " ", text)  # HTML Tags
    text = text.strip().lower()
    return text

In [12]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_data)
validation_data["summary"] = validation_data["summary"].apply(clean_data)

In [13]:
train_data["dialogue"][0]

"violet: hi! i came acro  thi  au tin'  article and i thought that you might find it intere ting violet:   claire: hi! :) thank , but i've already read it. :) claire: but thank  for thinking about me :)"

In [14]:
# Tokenizar
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [15]:
# row data => tokenized inputs for fine-tuning

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding="max_length", max_lenght = 512, truncation = True)
    targets = tokenizer(data["summary"], padding="max_length", max_lenght = 150, truncation = True)

    inputs["labels"] = targets["input_ids"]  # token id => add to input as labels
    return inputs

In [16]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
validation_dataset = validation_data.apply(tokenize, axis=1).tolist()


In [17]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 3, 9, 2771, 3, 7436, 185, 3, 17, 77, 31, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1413, 15, 3, 1222, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2763, 3, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2763, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [33]:
#  inputs_ids
# 1 => EOS    0 => Padding
#  attention_mask => those have 1 its mean this important and valid values other not imp during the training
#  labels - target => summary token

In [18]:
 #  Working with our Model
#NLP => generation task
model = T5ForConditionalGeneration.from_pretrained("t5-small")


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [19]:
import torch 
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device: ", device)
model.to(device)

Device:  mps


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments for Transformer

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,

    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",
    save_strategy = "epoch",

    warmup_steps = 500  # 0 => learning rate default
)

In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset
)

In [22]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [23]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [26]:
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # tokenize
    inputs = tokenizer(
        dialogue,
        padding="max_length",
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)

    # generate the summary => token ids
    model.to(device)
    targets = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=150,
        num_beams=4,
        early_stopping=True
    )
    
    # decoded our output
    summary = tokenizer.decode(targets[0], skip_special_tokens=True) # EOS, SEP
    return summary

In [27]:
test_dialogue = """ 
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""

summary = summarize_dialogue(test_dialogue)

print("Summary: ", summary)

Summary:  ai y tem are becoming more capable due to advance in deep learning and acce to large data et. the e model can now perform complex ta k uch a language under tanding, image recognition, and even code generation.
